In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")

In [ ]:
from utils import olc_client
c = olc_client.connect(verbose=True)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

import matplotlib as mpl
import matplotlib.pyplot as plt

from utils.ol_color import OL_COLOR
import cmap

import pandas as pd
import numpy as np
import re
import pickle
import alphashape
from scipy.stats import gaussian_kde
import os, datetime
# from openpyxl.styles import Font

import neuprint
from neuprint import fetch_neurons, NeuronCriteria as NC, SynapseCriteria as SC

import navis
import navis.interfaces.neuprint as neu

In [ ]:
from utils.config import CACHE_DIR, DATA_DIR, FIG_DIR

result_dir = FIG_DIR / 'quan_propagation'
result_dir.mkdir(parents=True, exist_ok=True)

cache_path = CACHE_DIR / 'connectivity'

# Neuropil rois

In [ ]:
rois_dict = neuprint.queries.fetch_roi_hierarchy(include_subprimary=False)

# combine keys in 'CentralBrain' and 'Optic(R)'
rois = set(rois_dict['CNS']['CentralBrain'].keys()) #| set(rois_dict['CNS']['Optic(R)'].keys())

rois = list(rois)
rois.sort()
print(len(rois))

# keep only primary rois
# remove strings that doesn't have * at the end
rois = [r for r in rois if '*' in r]
# rois = [r for r in rois if '*' not in r]
print(len(rois))

# remove * at the end of the string
rois = [re.sub(r'\*$', '', roi) for roi in rois]
# remove strings that contain 'unspecified' 
rois = [roi for roi in rois if 'unspecified' not in roi]
print(len(rois))

rois_cb = rois.copy()


### get brain mesh to set plotting ranges

In [ ]:
cb_mesh = neu.fetch_roi('CentralBrain')
olr_mesh = neu.fetch_roi('Optic(R)')
oll_mesh = neu.fetch_roi('Optic(L)')

In [ ]:
# check range
# xyz = np.vstack([cb_mesh.vertices, olr_mesh.vertices, oll_mesh.vertices])
# print(
#     np.max(xyz[:, 0]),
#     np.min(xyz[:, 0]),
#     np.max(xyz[:, 1]),
#     np.min(xyz[:, 1]),
#     np.max(xyz[:, 2]),
#     np.min(xyz[:, 2])
# )
    

In [ ]:
# zrange = [10000, 20000, 30000, 42000]
zrange = [10000, 21000, 42000] # central section
# zrange = [10000, 42000]

# # cb
# xrange = [20000, 80000]
# yrange = [4000, 55000]
# cns
# xrange = [2000, 95000]
xrange = [0000, 102000] # 2x width for kde
yrange = [4000, 55000]

### make brain and roi outlines

In [ ]:
from quan_propagation.func import get_roi_outline

In [ ]:
# Me, instead of whole OL
# outlines_bkgd = get_roi_outline(['ME(R)', 'CentralBrain', 'ME(L)'], 0.0002)
outlines_bkgd = get_roi_outline(['ME(R)', 'ME(L)'], 0.0002)
# xyz = neu.fetch_roi('ME(R)').vertices
# alpha_shape = alphashape.alphashape(xyz[:, :2], 0.0002)
# me_r_outline = [[x, y] for x, y in alpha_shape.exterior.coords]

# xyz = neu.fetch_roi('ME(L)').vertices
# alpha_shape = alphashape.alphashape(xyz[:, :2], 0.0002)
# me_l_outline = [[x, y] for x, y in alpha_shape.exterior.coords]

# outlines_bkgd = [me_r_outline, me_l_outline]

# specific
outlines_roi = get_roi_outline(['AOTU(R)', 'AOTU(L)', 'PLP(R)', 'SPS(R)', 'PVLP(L)', 'EB'], 0.0002)

In [ ]:
# relevant CB rois, with sectioning

# MeTu
# outlines_roi_sec = get_roi_outline(['AOTU(R)', 'AOTU(L)', 'BU(R)', 'BU(L)', 'EB'], 0.002, zranges=zrange)

# LP
outlines_roi_sec = get_roi_outline(['PLP(R)', 'IPS(R)', 'LAL(R)', 'PLP(L)', 'IPS(L)', 'LAL(L)'], 0.002, zranges=zrange)
# remove LAL for the first section
outlines_roi_sec[2][1] = []
outlines_roi_sec[5][1] = []


# LC
# # combine 'AVLP(R)', 'PVLP(R)',
# outlines_roi_sec = get_roi_outline(['AOTU(R)', 'PLP(R)',
#                                 'AOTU(L)', 'PLP(L)'
#                                 ], 0.002, zranges=zrange)
# # combine 'AVLP(R)', 'PVLP(R)', 
# xyz = [neu.fetch_roi(roi).vertices for roi in ['AVLP(R)', 'PVLP(R)']]
# xyz = np.vstack(xyz)
# outlines_tmp = []
# for j in range(len(zrange)-1):
#     xy = xyz[(xyz[:, 2] > zrange[j]) & (xyz[:, 2] < zrange[j+1]), :2]  # Select the first two columns (x and y)
#     alpha_shape = alphashape.alphashape(xy, alpha=0.002)
#     outlines_tmp.append([[x, y] for x, y in alpha_shape.exterior.coords])
# outlines_roi_sec.append(outlines_tmp)
# # left
# xyz = [neu.fetch_roi(roi).vertices for roi in ['AVLP(L)', 'PVLP(L)']]
# xyz = np.vstack(xyz)
# outlines_tmp = []
# for j in range(len(zrange)-1):
#     xy = xyz[(xyz[:, 2] > zrange[j]) & (xyz[:, 2] < zrange[j+1]), :2]  # Select the first two columns (x and y)
#     alpha_shape = alphashape.alphashape(xy, alpha=0.002)
#     outlines_tmp.append([[x, y] for x, y in alpha_shape.exterior.coords])
# outlines_roi_sec.append(outlines_tmp)


# Spatial resolution

In [ ]:
# Load
rf_summary = pd.read_csv(Path(result_dir, 'rf_summary.csv'))

In [ ]:
rf_summary.head()

# RI

## load retinotopy index

In [ ]:
# read csv file
ri0 = pd.read_csv(result_dir / 'vpn_retinotopy' / 'ri.csv', index_col=False)

In [ ]:
ri0.head()

In [ ]:
ri0 = ri0[ri0['syn_post_mean'] > 0.2]

## get mean syn positions

In [ ]:
# initialize empty df 
xyz_si_all = pd.DataFrame()

for i in range(len(ri0)):
    inst = ri0.iloc[i]['celltype'] + '_R'
    syn_pre = neuprint.fetch_mean_synapses(
        NC(instance=inst), SC(type='pre', rois='CentralBrain', primary_only=False)
    )
    # add column SI 
    syn_pre['SI'] = ri0.iloc[i]['syn_post_mean']
    # append to xyz_si_all
    xyz_si_all = pd.concat([xyz_si_all, syn_pre], ignore_index=True)


# plot


In [ ]:
#  bins
xnbins = 300
ynbins = 150
# xnbins = 4
# ynbins = 2
xi, yi = np.mgrid[
    xrange[0]:xrange[1]:xnbins*1j,
    yrange[0]:yrange[1]:ynbins*1j
    ]


In [ ]:
# in the meshgrid xi yi, in each tile, seach for rows in xyz_si_all that fall within that tile, and take the max SI values
zi = np.zeros(xi.shape)
for i in range(xi.shape[0]):
    for j in range(xi.shape[1]):
        # find rows in xyz_si_all that fall within the tile
        mask = (
            (xyz_si_all['x'] >= xi[i, j]) &
            (xyz_si_all['x'] < xi[i, j] + (xrange[1] - xrange[0]) / xnbins) &
            (xyz_si_all['y'] >= yi[i, j]) &
            (xyz_si_all['y'] < yi[i, j] + (yrange[1] - yrange[0]) / ynbins)
        )
        # take the max SI value
        if mask.any():
            zi[i, j] = xyz_si_all.loc[mask, 'SI'].abs().max()
        else:
            zi[i, j] = 0

In [ ]:
# smooth the zi
from scipy.ndimage import gaussian_filter
zcolor = gaussian_filter(zi, sigma=0.5)


In [ ]:

fig, ax = plt.subplots(figsize=(8,4))
ax.pcolormesh(xi, yi, zcolor)

# ol roi
for m in range(len(outlines_bkgd)):
    xy = np.array(outlines_bkgd[m])
    ax.plot(xy[:,0], xy[:,1], color='white', linewidth=1)

# specific roi
for m in range(len(outlines_roi)):
    xy = np.array(outlines_roi[m])
    ax.plot(xy[:,0], xy[:,1], color='white', linewidth=1)

ax.set_aspect('equal', adjustable='box')
# set xlim and ylim to the same range as xi and yi
ax.set_xlim(xrange[0], xrange[1])
ax.set_ylim(yrange[0], yrange[1])
# reverse the y axis
ax.invert_yaxis()

# colorbar
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label('SI', rotation=270, labelpad=15)


# save figure
plt.savefig(Path(result_dir, 'vpn_retinotopy',  f'RI_vpn.pdf'), dpi=300, bbox_inches='tight')


In [ ]:
#  composite color

# kde bins
xnbins = 300
ynbins = 150
xi, yi = np.mgrid[
    xrange[0]:xrange[1]:xnbins*1j,
    yrange[0]:yrange[1]:ynbins*1j
    ]

# color map
cmap = [cmp_red, cmp_green, cmp_blue] 

# number of hops, + 0 for LPT
nhop = 1 + 3

# one plot per section
for j in range(len(zrange)-1):
    zi_lst = []
    # first k hops
    for k in range(nhop):
        syn_xyz = syn_all[k]

        # collapse syn_xyz to 2D section
        xy = syn_xyz.loc[(syn_xyz.z > zrange[j]) & (syn_xyz.z < zrange[j+1]), ['x', 'y']]
        x = pd.to_numeric(xy['x'])  # Ensure x is numeric
        y = pd.to_numeric(xy['y'])  # Ensure y is numeric

        # kde
        gkde = gaussian_kde([x, y], bw_method= 0.1)
        zi = gkde(np.vstack([
            xi.flatten(),
            yi.flatten()
            ])).reshape(xi.shape)
        zi_lst.append(zi)

    # max and min of zi_lst
    zmax = [np.max(zi) for zi in zi_lst] 
    zmin = []
    for k in range(len(zi_lst)):
        zi = zi_lst[k]
        zsum = sum(zi.flatten())
        zsorted = np.sort(zi.flatten())
        # set zmin to be the bottom 1% in cumsum
        idx = np.argmax(np.cumsum(zsorted) > zsum * 0.01)
        zmin.append(zsorted[idx])
    print(zmin, zmax)

    zcolor = []
    for k in range(len(zi_lst)):
        # Log and normalize 
        norm = plt.cm.colors.LogNorm(vmin=zmin[k], vmax=zmax[k])
        zi = norm(zi_lst[k])
        zi[zi<0] = 0
        # assign color to each zi value
        zcolor.append(cmap[k](zi))

    # add up zcolor
    zcolor_sum = np.zeros(zcolor[0].shape)
    for k in range(len(zcolor)):
        zcolor_sum += zcolor[k]
    # set alpha to be 1
    zcolor_sum[:,:, 3] = 1

    # plot
    fig, ax = plt.subplots(figsize=(8,4))
    ax.pcolormesh(xi, yi, zcolor_sum)

    # - overlay the outline of the mesh on top of the density plot,
    # bkgd roi
    # section by zrange -> not good since the OLs are in the back of the brain
    # xy_ls = outlines_bkgd_sec[j]
    # for m in range(len(xy_ls)):
    #     xy = np.array(xy_ls[m])
    #     ax.plot(xy[:,0], xy[:,1], color='white', linewidth=1)
    # central section
    for m in range(len(outlines_bkgd)):
        xy = np.array(outlines_bkgd[m])
        ax.plot(xy[:,0], xy[:,1], color='white', linewidth=1)

    # specific roi
    for m in range(len(outlines_metu)):
        xy = np.array(outlines_metu[m])
        ax.plot(xy[:,0], xy[:,1], color='white', linewidth=1)

    # - some plotting parameters
    # aspect ratio = 1
    ax.set_aspect('equal', adjustable='box')
    # set xlim and ylim to the same range as xi and yi
    ax.set_xlim(xrange[0], xrange[1])
    ax.set_ylim(yrange[0], yrange[1])
    # reverse the y axis
    ax.invert_yaxis()

    # # colorbar
    # cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
    # save figure
    plt.savefig(Path(result_dir, f'tbar_kde_metu_sec_{j}_{nhop-1}hop.png'), dpi=300, bbox_inches='tight')